RF, Anti-CCP → impute + add _was_missing flag (their missingness was class-dependent)
Everything else (ESR, CRP, HLA-B27, ANA, Anti-Ro, Anti-La, Anti-dsDNA, Anti-Sm, C3, C4) → straight MICE, no flag needed

In [2]:
import pandas as pd
import numpy as np

df = pd.read_excel("../data/raw/dataset.xlsx")
df.shape

(12085, 15)

In [3]:
# add missing flags because we are adding aditional column of was this originally missing needed for rf
df['RF_was_missing'] = df['RF'].isna().astype(int)
df['Anti-CCP_was_missing'] = df['Anti-CCP'].isna().astype(int)

df[['RF', 'RF_was_missing', 'Anti-CCP', 'Anti-CCP_was_missing']].head(10)

,RF,RF_was_missing,Anti-CCP,Anti-CCP_was_missing
0,34.2,0,29.9,0
1,35.5,0,28.9,0
2,21.3,0,21.3,0
3,26.0,0,39.0,0
4,38.1,0,30.8,0
5,NaN,1,37.3,0
6,22.3,0,NaN,1
7,31.8,0,38.1,0
8,33.4,0,NaN,1
9,37.4,0,25.0,0


In [4]:
df['RF_was_missing'].sum(), df['RF'].isna().sum()

(np.int64(1329), np.int64(1329))

In [5]:
#MICE needs  umeric so first convert positive/negative to 1/0
binary_cols = ['HLA-B27', 'ANA', 'Anti-Ro', 'Anti-La', 'Anti-dsDNA', 'Anti-Sm']

for col in binary_cols:
    df[col] = df[col].map({'Positive': 1, 'Negative': 0})

df[binary_cols].head(10)

,HLA-B27,ANA,Anti-Ro,Anti-La,Anti-dsDNA,Anti-Sm
0,1.0,0.0,1.0,0.0,1.0,1.0
1,0.0,NaN,1.0,NaN,1.0,NaN
2,0.0,0.0,NaN,1.0,0.0,NaN
3,NaN,NaN,1.0,1.0,NaN,NaN
4,1.0,0.0,1.0,0.0,1.0,0.0
5,1.0,NaN,0.0,0.0,NaN,0.0
6,0.0,1.0,0.0,NaN,1.0,1.0
7,0.0,1.0,0.0,0.0,1.0,1.0
8,NaN,NaN,1.0,NaN,1.0,NaN
9,1.0,0.0,0.0,NaN,1.0,1.0


In [6]:
df['Gender'] = df['Gender'].map({'Male': 1, 'Female': 0})

In [7]:
%pip install scikit-learn imbalanced-learn pyarrow

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
from sklearn.experimental import enable_iterative_imputer 
from sklearn.impute import IterativeImputer

mice_cols = ['ESR', 'CRP', 'RF', 'Anti-CCP', 'HLA-B27', 'ANA',
             'Anti-Ro', 'Anti-La', 'Anti-dsDNA', 'Anti-Sm', 'C3', 'C4']

imputer = IterativeImputer(random_state=42, max_iter=15)
df[mice_cols] = imputer.fit_transform(df[mice_cols])

df[mice_cols].isna().sum()

c:\Users\hp\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\impute\_iterative.py:867: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(


ESR           0
CRP           0
RF            0
Anti-CCP      0
HLA-B27       0
ANA           0
Anti-Ro       0
Anti-La       0
Anti-dsDNA    0
Anti-Sm       0
C3            0
C4            0
dtype: int64

In [9]:
for col in binary_cols:
    df[col] = df[col].round().clip(0, 1)

df[binary_cols].describe()

,HLA-B27,ANA,Anti-Ro,Anti-La,Anti-dsDNA,Anti-Sm
count,12085.000000,12085.000000,12085.000000,12085.000000,12085.000000,12085.000000
mean,0.661316,0.650228,0.609516,0.624576,0.578072,0.565163
std,0.473282,0.476918,0.487879,0.484252,0.493888,0.495756
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
75%,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
max,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


In [10]:
continuous_cols = ['ESR', 'CRP', 'RF', 'Anti-CCP', 'C3', 'C4']
df[continuous_cols].describe()

,ESR,CRP,RF,Anti-CCP,C3,C4
count,12085.000000,12085.000000,12085.000000,12085.000000,12085.000000,12085.000000
mean,24.204050,13.354794,19.667341,19.625152,131.881520,38.171948
std,14.170279,10.101588,11.003112,10.263750,34.338357,18.629629
min,0.000000,-3.900161,0.000000,0.000000,50.000000,5.000000
25%,10.000000,2.100000,10.600000,12.100000,108.000000,24.000000
50%,28.000000,15.700000,19.200000,19.349439,133.000000,38.551214
75%,36.000000,21.966662,28.628886,27.100000,157.000000,52.389388
max,49.000000,31.024280,40.000000,40.000000,205.000000,74.000000


In [11]:
# clip any negative values to 0 for the continuous markers (RF, Anti-CCP already ≥0 as shown, but clip defensively)
for col in continuous_cols:
    df[col] = df[col].clip(lower=0)

df[continuous_cols].describe()

,ESR,CRP,RF,Anti-CCP,C3,C4
count,12085.000000,12085.000000,12085.000000,12085.000000,12085.000000,12085.000000
mean,24.204050,13.374947,19.667341,19.625152,131.881520,38.171948
std,14.170279,10.072846,11.003112,10.263750,34.338357,18.629629
min,0.000000,0.000000,0.000000,0.000000,50.000000,5.000000
25%,10.000000,2.100000,10.600000,12.100000,108.000000,24.000000
50%,28.000000,15.700000,19.200000,19.349439,133.000000,38.551214
75%,36.000000,21.966662,28.628886,27.100000,157.000000,52.389388
max,49.000000,31.024280,40.000000,40.000000,205.000000,74.000000


In [12]:
df[binary_cols].apply(pd.Series.value_counts)

,HLA-B27,ANA,Anti-Ro,Anti-La,Anti-dsDNA,Anti-Sm
1.0,7992,7858,7366,7548,6986,6830
0.0,4093,4227,4719,4537,5099,5255


In [13]:
df['Inflammation_Score'] = df['ESR'] + df['CRP']
df['C3_C4_Ratio'] = df['C3'] / df['C4']
df['Autoantibody_Count'] = df[binary_cols].sum(axis=1)

df[['Inflammation_Score', 'C3_C4_Ratio', 'Autoantibody_Count']].describe()

,Inflammation_Score,C3_C4_Ratio,Autoantibody_Count
count,12085.000000,12085.000000,12085.000000
mean,37.578997,4.708969,3.688871
std,23.601320,3.163125,1.377340
min,0.000000,1.229730,0.000000
25%,12.200000,2.666667,3.000000
50%,45.544071,3.551724,4.000000
75%,57.600000,5.600000,5.000000
max,80.024280,21.458869,6.000000


In [14]:
df.groupby('Disease')[['Inflammation_Score', 'C3_C4_Ratio', 'Autoantibody_Count']].mean().round(2)

,Inflammation_Score,C3_C4_Ratio,Autoantibody_Count
Disease,,,
Ankylosing Spondylitis,53.38,3.98,3.48
Normal,10.82,4.06,3.59
Psoriatic Arthritis,62.36,4.01,2.87
Reactive Arthritis,40.37,4.04,3.81
Rheumatoid Arthritis,55.36,4.08,3.24
Sjögren's Syndrome,10.49,3.99,4.68
Systemic Lupus Erythematosus,10.43,10.09,4.75


In [15]:
# label diseases using label encoder
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df['Disease_encoded'] = le.fit_transform(df['Disease'])

# check the mapping so we can decode predictions back to disease names later
dict(zip(le.classes_, le.transform(le.classes_)))

{'Ankylosing Spondylitis': np.int64(0),
 'Normal': np.int64(1),
 'Psoriatic Arthritis': np.int64(2),
 'Reactive Arthritis': np.int64(3),
 'Rheumatoid Arthritis': np.int64(4),
 "Sjögren's Syndrome": np.int64(5),
 'Systemic Lupus Erythematosus': np.int64(6)}

In [16]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Stage 1 feature set: original 14 raw features only
raw_features = ['Age', 'Gender', 'ESR', 'CRP', 'RF', 'Anti-CCP', 'HLA-B27',
                 'ANA', 'Anti-Ro', 'Anti-La', 'Anti-dsDNA', 'Anti-Sm', 'C3', 'C4']

# Stage 1b feature set: raw + missingness flags + engineered features
extended_features = raw_features + ['RF_was_missing', 'Anti-CCP_was_missing',
                                     'Inflammation_Score', 'C3_C4_Ratio', 'Autoantibody_Count']

y = df['Disease_encoded']

# split indices once, so train/test rows are IDENTICAL across both feature sets
# (critical for a fair apples-to-apples comparison)
X_raw = df[raw_features]
X_ext = df[extended_features]

X_raw_train, X_raw_test, y_train, y_test = train_test_split(
    X_raw, y, test_size=0.2, stratify=y, random_state=42
)

# use the SAME row indices for the extended set, so train/test rows match exactly
X_ext_train = X_ext.loc[X_raw_train.index]
X_ext_test = X_ext.loc[X_raw_test.index]

print(X_raw_train.shape, X_raw_test.shape)
print(X_ext_train.shape, X_ext_test.shape)

(9668, 14) (2417, 14)
(9668, 19) (2417, 19)


In production, your model receives a single new user, a new sensor ping, or a new transaction. You cannot calculate the mean or standard deviation of data you haven't received yet. Therefore, any step that uses the test set to determine how to process the training set cheats this simulation.

In [17]:
continuous_cols = ['Age', 'ESR', 'CRP', 'RF', 'Anti-CCP', 'C3', 'C4']
extended_continuous_cols = continuous_cols + ['Inflammation_Score', 'C3_C4_Ratio']

extended_continuous_cols = continuous_cols + ['Inflammation_Score', 'C3_C4_Ratio', 'Autoantibody_Count']

scaler_raw = StandardScaler()
X_raw_train_scaled = X_raw_train.copy()
X_raw_test_scaled = X_raw_test.copy()
X_raw_train_scaled[continuous_cols] = scaler_raw.fit_transform(X_raw_train[continuous_cols])
X_raw_test_scaled[continuous_cols] = scaler_raw.transform(X_raw_test[continuous_cols])

scaler_ext = StandardScaler()
X_ext_train_scaled = X_ext_train.copy()
X_ext_test_scaled = X_ext_test.copy()
X_ext_train_scaled[extended_continuous_cols] = scaler_ext.fit_transform(X_ext_train[extended_continuous_cols])
X_ext_test_scaled[extended_continuous_cols] = scaler_ext.transform(X_ext_test[extended_continuous_cols])

X_raw_train_scaled.head()

,Age,Gender,ESR,CRP,RF,Anti-CCP,HLA-B27,ANA,Anti-Ro,Anti-La,Anti-dsDNA,Anti-Sm,C3,C4
327,0.908935,0,-0.051554,-0.093818,0.728475,1.121494,0.0,1.0,0.0,1.0,1.0,0.0,0.067565,0.066177
10698,1.473599,0,1.676747,1.253853,-1.562233,-1.047476,0.0,1.0,1.0,0.0,1.0,0.0,0.678869,0.099228
6344,0.061939,1,-0.579142,-1.117981,1.276055,-0.113749,0.0,1.0,1.0,1.0,1.0,1.0,-0.211560,0.206758
4506,-1.462653,0,0.478306,1.198816,-0.786495,-0.211012,1.0,1.0,1.0,0.0,1.0,0.0,-0.601959,1.174526
5436,1.021867,0,0.760292,0.244256,0.819739,0.615726,1.0,1.0,0.0,1.0,1.0,1.0,-0.048874,0.529347


Cleaned, imputed data (MICE for most features, missing-indicator flags for RF/Anti-CCP)
Engineered features computed and held aside (Inflammation_Score, C3_C4_Ratio, Autoantibody_Count)
Two scaled feature sets with matching train/test rows: X_raw_train_scaled/X_raw_test_scaled (14 features, for Stage 1) and X_ext_train_scaled/X_ext_test_scaled (19 features, for the engineered-features comparison)
Encoded label y_train/y_test with the class mapping saved

In [18]:
import os

os.makedirs('../data/processed', exist_ok=True)

# raw feature set (Stage 1)
X_raw_train_scaled.to_csv('../data/processed/X_raw_train.csv', index=True)
X_raw_test_scaled.to_csv('../data/processed/X_raw_test.csv', index=True)

# extended feature set (Stage 1b comparison)
X_ext_train_scaled.to_csv('../data/processed/X_ext_train.csv', index=True)
X_ext_test_scaled.to_csv('../data/processed/X_ext_test.csv', index=True)

# labels (shared across both feature sets, same rows)
y_train.to_csv('../data/processed/y_train.csv', index=True)
y_test.to_csv('../data/processed/y_test.csv', index=True)

print("Saved all processed files to data/processed/")

Saved all processed files to data/processed/
